# Arete plate detector — training

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Runtime → Run all.** It takes about 45–75 minutes: making the pictures (~25 min), training (~20–40 min).
3. At the end your browser downloads **`plate-detector.onnx`**. Put it in your project's `models` folder (`jarvis/models/plate-detector.onnx`) and tell Claude.

Nothing from your gym videos is uploaded; the pictures are made by the app's own plate simulator.

In [ ]:
# The project (public on GitHub) and the tools
!rm -rf arete && git clone --depth 1 https://github.com/vietstetics/arete.git
%cd arete
!node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs)
!pip -q install onnx onnxruntime
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — set Runtime type to T4 GPU')

In [ ]:
# 20,000 training pictures with every plate marked (8 batches, made in parallel)
!mkdir -p data && seq 1 8 | xargs -P 4 -I{} node scripts/plate-detector/generate.mjs --n 2500 --seed {} --out data/shard-{}
!ls -lh data

In [ ]:
# Train, check on pictures it hasn't seen, export
!python scripts/plate-detector/train.py --data data --epochs 30 --batch 64 --out plate-detector.onnx

In [ ]:
# Download the model
from google.colab import files
files.download('plate-detector.onnx')